# Notebook 07: Tabular Foundation Models (TFMs) & Production Stacking (2022–2026)
## TabPFN, In-Context Learning & Multi-Level Ensembling with Treelite Serving

---

### 1. Executive Intuition & The 2022–2026 Paradigm Shift

#### The Mental Model: In-Context Tabular Learning
For forty years, machine learning followed a single universal workflow:
> **"Collect training data $\to$ Initialize random weights $\to$ Run iterative optimization (tree splits or backprop) $\to$ Save static model."**

Between 2022 and 2026, **Tabular Foundation Models (TFMs)** like **TabPFN (Nature, 2022 / v2/v3 2024–2026)** flipped this paradigm on its head:
> **"Treat the entire training dataset and query row as a prompt sequence to a pre-trained Transformer. Predict the answer in a single forward pass with zero training and zero hyperparameter tuning."**

In this capstone notebook, we explore:
1. **TabPFN**: How amortized Bayesian inference enables instant zero-shot tabular predictions.
2. **Multi-Level Stacking Ensembles**: Blending GBDTs with Tabular Foundation Models to achieve SOTA accuracy.
3. **Microsecond Inference with Treelite**: Compiling trained tree ensembles into standalone C code for microservice production APIs.

```mermaid
graph TD
    InputData[Customer Transaction Query] --> T1[XGBoost 2.x GPU]
    InputData --> T2[CatBoost GPU]
    InputData --> T3[LightGBM GBDT]
    InputData --> T4[TabPFN Zero-Shot TFM]
    T1 --> Out1[Prob Vector 1: 20 Classes]
    T2 --> Out2[Prob Vector 2: 20 Classes]
    T3 --> Out3[Prob Vector 3: 20 Classes]
    T4 --> Out4[Prob Vector 4: 20 Classes]
    Out1 & Out2 & Out3 & Out4 --> Meta[Level-2 Stacking Meta-Learner]
    Meta --> Final[Final 20-Class Ensembled Prediction]
    Final --> Treelite[Treelite C-Compiler: < 5us Serving]
```

--- 

### 2. Deep Mathematical Derivations

#### A. TabPFN: Amortized Bayesian Inference over Synthetic Priors
Traditional Bayesian tabular learning computes the posterior predictive distribution by integrating over parameter space $\theta$:
$$p(y_{\text{test}} \mid x_{\text{test}}, \mathcal{D}_{\text{train}}) = \int p(y_{\text{test}} \mid x_{\text{test}}, \theta) p(\theta \mid \mathcal{D}_{\text{train}}) d\theta$$
Because the integral is intractable, standard methods use Markov Chain Monte Carlo (MCMC), which takes hours.

**TabPFN replaces the integral with a Transformer** $q_\phi$ parameterized by weights $\phi$:
1. **Synthetic Prior Generation**: During a massive offline pre-training phase, TabPFN generates millions of synthetic datasets $\mathcal{D} = \{(x_1, y_1), \dots, (x_N, y_N)\}$ from causal Structural Equation Models (SEMs), random decision trees, and Gaussian Processes.
2. **Training Objective**: The Transformer is trained using cross-entropy to predict masked test labels given the remaining dataset as context:
   $$\min_\phi \mathbb{E}_{\mathcal{D} \sim P(\mathcal{D})} \left[ -\sum_{(x_i, y_i) \in \mathcal{D}_{\text{test}}} \ln q_\phi(y_i \mid x_i, \mathcal{D}_{\text{train}}) \right]$$
3. **In-Context Inference**: When you call `tabpfn.predict(X_test)` on our e-commerce dataset, **no optimization occurs**. The model simply passes $\mathcal{D}_{\text{train}}$ and $X_{\text{test}}$ through its pre-trained self-attention layers, computing the exact Bayesian posterior prediction in a **single forward pass ($O(1)$ iterations)**!

#### B. The Mathematics of Stacking Ensembles (Wolpert Super Learner)
Why does stacking diverse models beat individual tuned models?
Let $h_1(x), \dots, h_M(x)$ be $M$ diverse Level-1 models (e.g. XGBoost, LightGBM, CatBoost, TabPFN). Each outputs a 20-dimensional probability vector $z_{ik} = [h_1^k(x_i), \dots, h_M^k(x_i)]$.

##### Out-of-Fold (OOF) Cross-Validation Leakage Protection
If we train Level-1 models on the full dataset and feed their predictions to the Level-2 meta-learner, the meta-learner severely overfits because the predictions are overconfident.
**The Super Learner Protocol**:
1. Divide the training set into $K$ folds (e.g. 5 folds).
2. For each fold $k$, train Level-1 models on the remaining $K-1$ folds, and predict on fold $k$.
3. Concatenate these out-of-fold predictions to form a clean matrix $Z \in \mathbb{R}^{N \times 20M}$.
4. Train a regularized Level-2 Logistic Regression meta-learner on $Z$:
   $$\hat{y} = \text{Softmax}\left( W \cdot Z + b \right) \quad \text{subject to } L_2 \text{ penalty } \|W\|_F^2$$
Because tree ensembles (axis-aligned splits) and foundation models (self-attention) have orthogonal inductive biases, their prediction errors are nearly uncorrelated, **boosting the overall ensemble AUC beyond any individual model**.

In [ ]:
import sys, os
cur_dir = os.path.abspath(os.getcwd())
trees_dir = os.path.abspath(os.path.join(cur_dir, '..')) if os.path.basename(cur_dir) == 'notebooks' else cur_dir
repo_root = os.path.abspath(os.path.join(trees_dir, '..'))
for p in [trees_dir, repo_root]:
    if p not in sys.path: sys.path.insert(0, p)
# Setup environment and load utilities
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

if root_dir not in sys.path:

from trees.utils import (
    print_hardware_summary, load_dataset, prepare_features,
    evaluate_multiclass_model, plot_confusion_matrix_20, plot_metrics_comparison,
    DEPARTMENTS_EN
)

print_hardware_summary()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Load 10,000 samples for Foundation Model & Stacking Benchmark
df = load_dataset(sample_rows=10_000)
X_raw, y, cat_cols, num_cols = prepare_features(df)

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded = X_raw.copy()
X_encoded[cat_cols] = encoder.fit_transform(X_raw[cat_cols])

# 70/10/20 Stratified Split
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.6667, random_state=42, stratify=y_temp)

print(f"Train instances: {len(X_train):,} | Test instances: {len(X_test):,}")

### 3. Model 1: TabPFN Zero-Shot Tabular Inference
We evaluate TabPFN, demonstrating instant in-context learning without iterative gradient optimization.

In [ ]:
all_results = []

try:
    from tabpfn import TabPFNClassifier
    
    device_target = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    t0 = time.time()
    tabpfn = TabPFNClassifier(device=device_target, N_ensemble_configurations=8)
    # Note: 'fit' in TabPFN is instant (stores context in memory)
    tabpfn.fit(X_train.iloc[:1500], y_train[:1500])
    pfn_time = time.time() - t0
    
    pfn_pred = tabpfn.predict(X_test.iloc[:500])
    pfn_prob = tabpfn.predict_proba(X_test.iloc[:500])
    pfn_metrics = evaluate_multiclass_model("TabPFN (Zero-Shot Tabular Foundation Model)", y_test[:500], pfn_pred, pfn_prob, pfn_time)
    all_results.append(pfn_metrics)
    print("TabPFN Evaluation:", pfn_metrics)
except ImportError:
    print("tabpfn not installed; skipping TabPFN benchmark.")

### 4. Model 2: Multi-Level Production Stacking Ensemble
We build an enterprise-grade Stacking Ensemble blending **XGBoost 2.x**, **LightGBM**, and **CatBoost** via a Level-2 Ridge Logistic Regression meta-learner.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

device_xgb = 'cuda' if torch.cuda.is_available() else 'cpu'

estimators = [
    ('xgb', xgb.XGBClassifier(n_estimators=50, max_depth=5, learning_rate=0.1, tree_method='hist', device=device_xgb, random_state=42)),
    ('lgb', lgb.LGBMClassifier(n_estimators=50, max_depth=5, learning_rate=0.1, random_state=42, verbose=-1)),
    ('cat', CatBoostClassifier(iterations=50, depth=5, learning_rate=0.1, random_seed=42, verbose=False))
]

t0 = time.time()
stacking_ensemble = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=200, C=1.0),
    cv=3,
    n_jobs=1
)
stacking_ensemble.fit(X_train, y_train)
stack_time = time.time() - t0

stack_pred = stacking_ensemble.predict(X_test)
stack_prob = stacking_ensemble.predict_proba(X_test)
stack_metrics = evaluate_multiclass_model("Multi-Level Stacking Ensemble (XGB+LGB+CAT)", y_test, stack_pred, stack_prob, stack_time)
all_results.append(stack_metrics)

pd.DataFrame(all_results)[['Model', 'Training Time (s)', 'Top-1 Accuracy', 'Top-3 Accuracy', 'Multi-Class Log-Loss', 'Macro F1-Score']]

### 5. Compiling Trees to Low-Latency C Code (Treelite / ONNX)
Compiling trained tree ensembles into standalone C code for microsecond microservice serving.

In [ ]:
try:
    import treelite
    
    fast_xgb = xgb.XGBClassifier(n_estimators=30, max_depth=4, random_state=42)
    fast_xgb.fit(X_train, y_train)
    
    tl_model = treelite.Model.from_xgboost(fast_xgb.get_booster())
    print("Successfully compiled XGBoost model into Treelite representation!")
    print(f"Compiled Tree Model: {tl_model.num_tree} trees across 20 classes.")
except ImportError:
    print("treelite not installed; skipping C compilation demonstration.")

### 6. Master Summary: The Complete 2026 Tabular Decision Matrix

| Scenario | Recommended Model | Rationale |
|---|---|---|
| **Extreme Latency (< 10 μs)** | Google YDF or Treelite C-Compiled GBDT | Zero Python interpreter overhead, CPU SIMD optimization. |
| **Large-Scale Tabular (> 50k rows)** | LightGBM or XGBoost 2.x on GPU | Unmatched training throughput and low memory footprint. |
| **Heavy Categorical & Bilingual Data** | CatBoost (GPU) | Oblivious trees + ordered target statistics eliminate leakage. |
| **Regulated Systems (Credit, Health, Audit)**| EBM (Explainable Boosting Machine) | Exact $\text{GA}^2\text{M}$ glass-box additive curves. |
| **Continuous Streaming (Kafka/Flink)** | Hoeffding Trees (`river`) | Online sample-by-sample learning with zero retraining. |
| **Small Data (< 10k rows) & Cold Start**| TabPFN / Tabular Foundation Models | Zero tuning, instant Bayesian in-context predictions. |
| **Competitive Leaderboard (Kaggle/Max Acc)**| Multi-Level Stacking Ensemble | Blending GBDT + TFM predictions eliminates individual model blindspots. |